# Watt The Hack: Controller Training Starter (v0.3.0)

<a href="https://colab.research.google.com/github/AaronEliasZachariah/City-of-Melbourne-Watt-the-Hack-Advanced-Track/blob/main/notebooks/training_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Welcome to the advanced track! This notebook shows you how to **write and locally playtest** a controller for the Watt The Hack engine — against the **real scenarios**, with one function call. No hand-built state, no virtualenv, no command line.

When you're happy with it, submit through the **Submission Portal** on `mlai.au` (last section): paste your code in and hit submit. No zip, no API to call by hand.

The full reference is in the in-app docs: **Watt The Hack -> Docs**. This notebook mirrors them.

## How the loop works (read this first)

You write a **controller**. The engine runs the grid as a simulation in 15-minute steps (`duck_curve` runs three days — 288 steps). Each step, it hands your controller a snapshot of the grid and asks for an action:

```
every 15 minutes:

   state  ->  your controller  ->  action  ->  engine simulates  ->  cost
  (demand, solar,   step(state)     (battery,    physics + market
   price, soc, ...)                  diesel, ...)

   score  =  sum of cost over every step       (lower wins)
```

You see only the **current** step (plus a forecast in later scenarios) and return how much to charge/discharge the battery, run diesel, or curtail solar. The engine simulates that 15 minutes — clipping to physical limits, then applying the market — and charges you a cost. Your score is the total over every step.

**One worked step — the duck curve at midday.** Solar is 80 MW, demand 30 MW: a 50 MW surplus. Do nothing and that surplus floods the grid past its 50 MW export cap, and you eat an *overvoltage penalty*. Charge the battery instead (`battery_flow_mw = -20`) and you bank cheap midday energy to spend at the evening peak, when grid power is dear. *Store the midday glut, spend it at the evening peak* — that trade-off is the duck curve.

When you run `run_playtest` (section 3), it prints your cost next to a do-nothing and a naive baseline — so you always know whether a change actually helped. (The optimum is deliberately not shown; drive the cost down and watch the levers.)

## 1. Install the engine (with the playtest harness)

Install the public engine from **PyPI**. The `[playtest]` extra adds `matplotlib` (plots) and `openai` (the agentic track). Re-run with `--upgrade` after new scenarios drop.

The wheel bundles the playtest harness **and** the scenarios you can run offline: `duck_curve` and `agentic_demo`. (The scored judging variants stay on the server.)

In [ ]:
!pip install "watt-the-hack[playtest]"

### Colab vs. local — two ways to playtest

You're in **Colab** right now: zero setup, runs in the cloud, and the `!pip install` above is all you need. Best for starting fast.

To iterate harder — your own editor and debugger, instant re-runs, no cell shuffling — playtest **locally** on your machine instead. Same engine, same scenarios, same scores; only the setup differs:

| | **Colab** (here) | **Local** (your machine) |
|---|---|---|
| Setup | none | a virtual env + one `pip install` |
| Install | `!pip install "watt-the-hack[playtest]"` | `python -m pip install "watt-the-hack[playtest]"` — inside an **activated venv** |
| Run | `run_playtest("strategy.py", "duck_curve")` in a cell | `python strategy.py` (a one-file controller with a `__main__` run block) — or the IDE ▶ Run button |

It's the same `pip install watt-the-hack` either way. **The full local walkthrough — venv setup and the self-running `strategy.py` — is in the public engine repo's README:**

### 👉 https://github.com/AaronEliasZachariah/City-of-Melbourne-Watt-the-Hack-Advanced-Track

A green run locally is identical to a green run here — and to what the submission portal scores. Many people prototype in Colab, then move local once they're iterating hard.

## 2. The state -> action contract

Every step, your controller receives a `state` dict and returns an `action` dict.

**Key things you read from `state`:**
- `time`: timestep index
- `demand`, `solar`: current MW (current step only)
- `soc`: battery state of charge, 0.0 to 1.0
- `price`: current import tariff ($/MWh, roughly -50 to 550: cheap midday, dear in the evening peak)
- `forecast`: lookahead arrays (`demand`, `solar`, `price`) for horizon planning
- `features`: which mechanics are active (e.g. `{"battery": true, "fcas": false}`)
- `alerts`: qualitative text events (operator directives, cyber alerts) for agentic scenarios

**Keys you return in `action`** (anything omitted defaults to 0):
- `battery_flow_mw`: **positive = discharge** to grid, **negative = charge** from grid
- `emergency_generator`: MW of diesel to dispatch
- `curtail_solar`: MW of solar to disconnect (prevents overvoltage)
- `fcas_reserve_mw`: MW of inverter capacity reserved for frequency control

> The inverter limit is shared between `battery_flow_mw` and `fcas_reserve_mw`: reserving FCAS reduces how fast you can charge or discharge that step.

## 3. Playtest in one call

Here's the whole loop. Write your controller to a file, then run it against a real scenario with `run_playtest`. You get the same score, cost breakdown, and per-step trace the **cloud judge** produces — it's the *same harness*: same filtered `state` view, same `plan` / `replan` / `step` loop.

`run_playtest(controller_path, scenario_id)` — scenarios bundled in the wheel: `duck_curve`, `agentic_demo`.

> `result["metrics"]["final_score"]` is your **raw cost in dollars — lower wins.** (The 0-150 leaderboard points are computed server-side on the hidden judging variants; locally you optimise the raw cost.)

In [ ]:
%%writefile strategy.py
# Your controller is EITHER a controller(state) function OR a Strategy class (see section 6).
# Start with the simplest possible thing: do nothing.
def controller(state):
    # battery_flow_mw: positive = discharge to grid, negative = charge from grid
    return {"battery_flow_mw": 0.0, "emergency_generator": 0.0, "curtail_solar": 0.0}

In [ ]:
from watt_the_hack.playtest import run_playtest

result = run_playtest("strategy.py", "duck_curve", plots=False)
print(f"\nRaw cost (lower wins): ${result['metrics']['final_score']:,.2f}")

## 4. A rule-based controller

Charge when power is cheap, discharge when it's expensive. Same flow: write it, run it, read the cost breakdown to see where the money went.

In [ ]:
%%writefile strategy.py
def controller(state):
    price = state.get("price", 0.0)
    soc = state.get("soc", 0.0)
    action = {"emergency_generator": 0.0, "curtail_solar": 0.0, "fcas_reserve_mw": 0.0}
    if price < 100 and soc < 0.9:
        action["battery_flow_mw"] = -20.0   # charge when cheap
    elif price > 300 and soc > 0.1:
        action["battery_flow_mw"] = 20.0    # discharge when expensive
    else:
        action["battery_flow_mw"] = 0.0
    return action

In [ ]:
result = run_playtest("strategy.py", "duck_curve", plots=False)
print(f"\nRaw cost (lower wins): ${result['metrics']['final_score']:,.2f}\n")

# Where the money went (negative = revenue):
breakdown = {k: v for k, v in result["breakdown"].items() if abs(v) > 1e-6 and k != "total"}
for k, v in sorted(breakdown.items(), key=lambda kv: -abs(kv[1])):
    print(f"  {k:24s} ${v:>14,.2f}")

## 5. Inline plots (optional)

`result["rows"]` holds the full per-step trace, so you can plot inline with no files. (Needs `matplotlib`, included in the `[playtest]` extra. `run_playtest` also writes PNGs + a `report.html` to `runs/<scenario>_<timestamp>/` when you pass `plots=True`.)

In [ ]:
import matplotlib.pyplot as plt

rows = result["rows"]
t = [r["time_hours"] for r in rows]

fig, ax = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
ax[0].plot(t, [r["soc"] for r in rows], color="tab:blue")
ax[0].set_ylabel("SOC"); ax[0].set_ylim(0, 1); ax[0].grid(alpha=0.3)
ax[1].plot(t, [r["demand"] for r in rows], label="demand", color="tab:orange")
ax[1].plot(t, [r["solar"] for r in rows], label="solar", color="tab:green")
ax[1].plot(t, [r["net_grid_power_mw"] for r in rows], label="net grid", color="tab:gray")
ax[1].set_ylabel("MW"); ax[1].legend(loc="upper right"); ax[1].grid(alpha=0.3)
ax[2].plot(t, [r["cum_cost"] for r in rows], color="tab:red")
ax[2].set_ylabel("cumulative $"); ax[2].set_xlabel("hours"); ax[2].grid(alpha=0.3)
fig.suptitle(f"duck_curve - raw cost ${result['metrics']['final_score']:,.0f}")
fig.tight_layout(); plt.show()

## 6. The two submission shapes

Your submission is **one** of these. The Portal (and `run_playtest`) auto-detects which.

**Shape 1: a `controller(state)` function.** Use it when each step depends only on the current state (no memory between steps).

```python
def controller(state):
    return {"battery_flow_mw": state["demand"] - state["solar"]}
```

**Shape 2: a `Strategy` class.** Use it when you need to persist state between steps (a rolling buffer, a PID memory, an LLM plan). The engine instantiates it once and reuses it.

Requirements (enforced by the engine):
- The class **must** be named `Strategy`.
- It **must** define `step(self, state)` returning an action dict. *No `step` method means the submission is refused.*
- It must be constructible with no arguments.
- `plan(self, state)` and `replan(self, state, alerts)` are **optional** (used for agentic scenarios — section 8).

In [ ]:
%%writefile strategy.py
# Shape 2 skeleton: persists state across steps via self.*
class Strategy:
    def __init__(self):
        self.history = []          # anything you want to remember

    def step(self, state):          # REQUIRED
        self.history.append(state["soc"])
        price, soc = state["price"], state["soc"]
        flow = -20.0 if (price < 100 and soc < 0.9) else (20.0 if price > 300 else 0.0)
        return {"battery_flow_mw": flow, "emergency_generator": 0.0,
                "curtail_solar": 0.0, "fcas_reserve_mw": 0.0}

In [ ]:
result = run_playtest("strategy.py", "duck_curve", plots=False)
print(f"\nRaw cost (lower wins): ${result['metrics']['final_score']:,.2f}")

## 7. (Optional) Bake an ML model into your controller

The cloud evaluator's base image already ships `numpy`, `scipy`, `pandas`, `openai`, and `pydantic`, so you **can** `import numpy` in your submission; add anything else to `requirements.txt`.

If you'd rather ship **zero dependencies**, you can bake a trained model's weights directly into the code as plain Python literals (handy for tiny linear policies). The cell below demonstrates that export.

In [ ]:
import numpy as np

class SimpleModel:
    """A dummy model to demonstrate weight extraction."""
    def __init__(self):
        # Random weights for illustration: 4 actions based on 3 state variables
        self.weights = np.random.randn(4, 3)

    def predict(self, obs):
        return np.dot(self.weights, obs)

model = SimpleModel()

In [ ]:
def export_weights_to_code(model, filename="strategy.py"):
    weights_list = model.weights.tolist()

    code = f"""# Auto-generated submission (zero-dependency, weights baked in)
WEIGHTS = {weights_list}

def controller(state):
    obs = [state.get("soc", 0), state.get("price", 0), state.get("demand", 0)]
    actions = [sum(w * o for w, o in zip(row, obs)) for row in WEIGHTS]
    return {{
        "battery_flow_mw": max(-50.0, min(50.0, actions[0])),
        "emergency_generator": max(0.0, min(50.0, actions[1])),
        "curtail_solar": max(0.0, min(state.get("solar", 0.0), actions[2])),
        "fcas_reserve_mw": max(0.0, min(50.0, actions[3])),
    }}
"""
    with open(filename, "w") as f:
        f.write(code)
    print(f"Wrote {filename}. Playtest it, then paste its contents into the Submission Portal.")

export_weights_to_code(model)
with open("strategy.py") as f:
    print(f.read())

## 8. Agentic strategies with an LLM

Some scenarios deliver **qualitative text events** — operator directives, cyber alerts, policy changes — in `state["alerts"]`. There's no neat numeric forecast for these; they're English. You read the prose with a `Strategy` class and an LLM, translate it into a target, and act on it.

**The loop (this is the whole contract):**
- `plan(state)` runs **once** before step 0 — read the opening briefing from `state["alerts"]` here. The right place for a slow LLM call.
- `replan(state, alerts)` runs **only when a new alert fires** — re-read and update your plan.
- `step(state)` runs **every timestep** — read your cached target and act. **Never call the LLM here.**

> **The time budget (read this).** Your *entire* run must finish within about **14 minutes** of wall-clock. There's no per-step rescue: if the whole run goes over it ends in `TIMEOUT` with **no score** (a free retry — it doesn't burn one of your 3 attempts, but you get nothing back). One LLM call in `step()` is multiplied across every 15-minute tick and blows the budget. Call the LLM in `plan()` / `replan()` only, and use **`gpt-5.4-nano`**: it's fast, and it draws on a shared credit pool, so keep calls few.

**LLM access — local vs the judging platform:**
- **Locally** (this notebook) *you* set `OPENAI_API_KEY` (next cell). **On the judging platform it's injected for you** as an environment variable. The **same `os.environ` read works in both places**, so your `Strategy` code is identical local and submitted.
- Agentic scenarios run with outbound HTTPS egress, so the standard `OpenAI()` client reaches the API. (Non-agentic scenarios run network-isolated.)
- **Remove any `load_dotenv()` / `.env` loading before you submit**, and drop `python-dotenv` from `requirements.txt`. The platform ships no `.env`; a missing one silently leaves the key blank, which throws an `AuthenticationError` on the first call.
- Never hardcode a key in `strategy.py`.

Below we playtest a real agentic loop locally against **`agentic_demo`** — a one-day teaching scenario that issues a battery-reserve directive, **revises it mid-morning**, and throws in a vendor newsletter as noise. Your LLM has to obey the directive (and ignore the noise) or eat a reserve-shortfall penalty.

In [ ]:
import os

# Local / Colab: set your OWN key here. On the judging platform OPENAI_API_KEY is injected for you.
# In Colab, store it in the Secrets panel (the key icon, left sidebar) as OPENAI_API_KEY:
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    # Running locally (not Colab): paste your key, or set it in your shell before launching Jupyter.
    os.environ.setdefault("OPENAI_API_KEY", "sk-...replace-me...")

print("OPENAI_API_KEY set:", os.environ.get("OPENAI_API_KEY", "").startswith("sk-"))

In [ ]:
%%writefile strategy.py
import json
from openai import OpenAI

SYSTEM = """You translate ONE grid-operator alert into a battery reserve target.
Reply with JSON only: {"reserve_soc": <number between 0 and 1, or null>}
- If the alert sets, raises, or lowers a state-of-charge reserve floor, return it as a
  fraction (e.g. "seventy percent" -> 0.7).
- If the alert is a newsletter, advert, or anything that is NOT a binding directive,
  return {"reserve_soc": null}."""


class Strategy:
    def __init__(self):
        # The platform injects OPENAI_API_KEY; locally you set it yourself.
        # Do NOT call load_dotenv() here — remove any .env loading before submitting.
        self.client = OpenAI()
        self.reserve_soc = 0.0      # current cached reserve target

    def _read_alert(self, alert):
        text = f"{alert.get('title', '')}\n{alert.get('description', '')}"
        resp = self.client.chat.completions.create(
            model="gpt-5.4-nano",
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": text}],
            response_format={"type": "json_object"},
            temperature=0.0,
        )
        val = json.loads(resp.choices[0].message.content or "{}").get("reserve_soc")
        if isinstance(val, (int, float)):
            self.reserve_soc = float(val)   # a real directive updates the target

    def plan(self, state):
        # Runs ONCE before step 0: read the opening briefing from the alerts.
        for alert in state.get("alerts", []):
            self._read_alert(alert)
        return {}

    def replan(self, state, alerts):
        # Runs only when NEW alerts fire: re-read and update the cached target.
        for alert in alerts:
            self._read_alert(alert)
        return {}

    def step(self, state):
        # Runs every timestep. NO LLM CALLS — just act on the cached target.
        soc = state.get("soc", 0.5)
        flow = -40.0 if soc < self.reserve_soc else 0.0   # charge toward reserve, else hold
        return {"battery_flow_mw": flow, "emergency_generator": 0.0,
                "curtail_solar": 0.0, "fcas_reserve_mw": 0.0}

In [ ]:
# Needs OPENAI_API_KEY set (cell above). Calls gpt-5.4-nano a handful of times (plan + replans).
result = run_playtest("strategy.py", "agentic_demo", plots=False)
print(f"\nRaw cost (lower wins): ${result['metrics']['final_score']:,.2f}")
print(f"reserve-shortfall penalty: ${result['breakdown'].get('compliance_penalty', 0.0):,.2f}"
      "  (~0 if you held the reserve through the evening window)")

## 9. Submitting your controller

Submit through the **Submission Portal** in the app:

> **mlai.au -> Watt The Hack -> City of Melbourne (Advanced) Track -> Submission Portal**

1. **Paste your `strategy.py` contents** into the code editor. No zip, no CLI; the portal packages and uploads it for you.
2. Add any extra pip packages to the **requirements** box (most submissions leave it empty; `numpy`/`scipy`/`pandas`/`openai` are already in the base image). Drop `python-dotenv` if you used it locally.
3. Pick the scenario and submit. The portal auto-detects whether you wrote a `controller` function or a `Strategy` class.

**Limits (enforced server-side):**
- **3 submissions per scenario.** The **Gauntlet allows 1** (it's 3x weighted, so make it count).
- A short **submission cooldown** may apply between attempts; the portal shows a countdown when it does.
- The portal tracks evaluation live and updates the leaderboard when scoring finishes.

A local green run translates directly to a submission — the portal runs the exact same controller shape. Run the cell below to write a final `strategy.py`, then copy its contents into the portal editor.

In [ ]:
%%writefile strategy.py
def controller(state):
    # Charge when power is cheap, discharge when it's expensive.
    price = state.get("price", 0.0)
    soc = state.get("soc", 0.0)
    action = {"emergency_generator": 0.0, "curtail_solar": 0.0, "fcas_reserve_mw": 0.0}
    if price < 120 and soc < 0.95:
        action["battery_flow_mw"] = -50.0   # Charge
    elif price > 250 and soc > 0.05:
        action["battery_flow_mw"] = 50.0    # Discharge
    else:
        action["battery_flow_mw"] = 0.0
    return action